# 第2章　チーム開発とコードの再現性 ― Git/GitHub・レビュー・CI

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 2.1　ブランチとプルリクエスト ― 壊さずに改善する

```bash
git switch -c feature/tversky-loss   # 作業用ブランチを作って切り替える
# （コードを編集・コミット）
git push -u origin feature/tversky-loss   # 自分のブランチをGitHubへ
```

## 版管理を3層で回す ― コード・データ・実験

```bash
dvc init
dvc add data/train_v3            # 実体はストレージへ、data/train_v3.dvc がGitに残る
git add data/train_v3.dvc .gitignore
dvc remote add -d storage /mnt/secure/dvc-store   # 院内の安全な保管先
dvc push                          # 実体を保管先へ
```

In [ ]:
import mlflow
with mlflow.start_run(run_name="unet_tversky_v3"):
    mlflow.log_params({"loss": "tversky", "alpha": 0.7, "lr": 1e-4,
                       "data_version": "train_v3", "git_sha": "a1b2c3d", "seed": 42})
    # ... 学習 ...
    mlflow.log_metrics({"val_dice": 0.71, "val_sens": 0.94})
    mlflow.log_artifact("outputs/confusion_matrix.png")
    mlflow.pytorch.log_model(model, "model")

## CIを医療AI向けに組む ― GitHub Actionsの実装

```yaml
# .github/workflows/ci.yml
name: ci
on: [pull_request]
jobs:
  guard:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with: {fetch-depth: 0}
      # 1) 患者データ・大容量ファイルの混入を止める
      - name: block PHI and large files
        run: |
          BASE="origin/${{ github.base_ref }}"
          if git diff --name-only "$BASE"... | grep -Ei '\.(dcm|nii|nii\.gz|pth|ckpt)$'; then
            echo "::error::DICOM/NIfTI/重みファイルはコミット禁止（DVCへ）"; exit 1
          fi
          # 必ず if 文で書くこと。`A && B && { ... }` と並べると、5MB超が「無い」ときに
          # 最後のテストが偽を返し、その終了ステータスがループ＝ステップの終了コードに
          # なって、正常なPRのたびにCIが無言で落ちる（run: の既定シェルは bash -e）。
          for f in $(git diff --name-only "$BASE"...); do
            if [ -f "$f" ] && [ "$(wc -c <"$f")" -gt 5000000 ]; then
              echo "::error::5MB超: $f"; exit 1
            fi
          done
      # 2) 体裁チェックと型
      - uses: actions/setup-python@v5
        with: {python-version: "3.11"}
      - run: pip install -r requirements-dev.txt
      - run: ruff check . && mypy src
      # 3) 中核関数のテスト（壊れると誤結果を生む部分）
      - run: pytest tests/ -q
```